In [43]:
# Import libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

In [44]:
# Load train and test data
train = pd.read_csv("datasets/mortgage_train.csv")
test = pd.read_csv("datasets/mortgage_test.csv")

y_train, y_test = train["default"].values, test["default"].values
X_train = train[["DTI", "LTV", "Score", "UnempShock", "PriorDelinq"]]
X_test = test[["DTI", "LTV", "Score", "UnempShock", "PriorDelinq"]]

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (250, 6), Test: (100, 6)


In [45]:
# Standardize continuous variables for gradient descent convergence
cont_cols = ["DTI", "LTV", "Score"]
means, stds = X_train[cont_cols].mean(), X_train[cont_cols].std()

X_train_sc = X_train.copy()
X_test_sc = X_test.copy()
X_train_sc[cont_cols] = (X_train[cont_cols] - means) / stds
X_test_sc[cont_cols] = (X_test[cont_cols] - means) / stds

X_train_np = np.column_stack([np.ones(len(X_train_sc)), X_train_sc.values])
X_test_np = np.column_stack([np.ones(len(X_test_sc)), X_test_sc.values])

In [46]:
# Define sigmoid and gradient descent functions for logistic regression
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def logistic_gd(X, y, lr=0.1, n_iter=50000, tol=1e-7):
    n, k = X.shape
    beta = np.zeros(k)
    for i in range(n_iter):
        p = sigmoid(X @ beta)
        grad = X.T @ (p - y) / n
        beta -= lr * grad
        if np.linalg.norm(grad) < tol:
            print(f"Converged at iteration {i}")
            break
    loss = -np.mean(y * np.log(p + 1e-10) + (1 - y) * np.log(1 - p + 1e-10))
    return beta, loss

In [47]:
# Part (i): Estimate logistic regression using gradient descent
beta_gd, loss_gd = logistic_gd(X_train_np, y_train)
print(f"Final loss: {loss_gd:.6f}\n")

names = ["Intercept", "DTI", "LTV", "Score", "UnempShock", "PriorDelinq"]
print("Gradient Descent Coefficients:")
for n, c in zip(names, beta_gd):
    print(f"  {n}: {c:.4f}")

Converged at iteration 10764
Final loss: 0.313874

Gradient Descent Coefficients:
  Intercept: -2.5662
  DTI: 0.3624
  LTV: 0.6386
  Score: 0.0617
  UnempShock: 0.7565
  PriorDelinq: 1.1064


In [48]:
# Part (i): Estimate logistic regression using statsmodels
X_train_sm = sm.add_constant(X_train_sc)
logit_model = sm.Logit(y_train, X_train_sm).fit(disp=0)
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  250
Model:                          Logit   Df Residuals:                      244
Method:                           MLE   Df Model:                            5
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.1049
Time:                        18:53:23   Log-Likelihood:                -78.468
converged:                       True   LL-Null:                       -87.669
Covariance Type:            nonrobust   LLR p-value:                  0.002483
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -2.5662      0.284     -9.020      0.000      -3.124      -2.009
DTI             0.3624      0.216      1.682      0.093      -0.060       0.785
LTV             0.6386      0.230      2.771    

In [49]:
# Part (i): Compare gradient descent and statsmodels coefficients
comparison = pd.DataFrame({
    "GD": beta_gd,
    "Statsmodels": logit_model.params.values,
    "Difference": beta_gd - logit_model.params.values
}, index=names)
print(comparison.round(4))

                 GD  Statsmodels  Difference
Intercept   -2.5662      -2.5662         0.0
DTI          0.3624       0.3624         0.0
LTV          0.6386       0.6386        -0.0
Score        0.0617       0.0617        -0.0
UnempShock   0.7565       0.7565        -0.0
PriorDelinq  1.1064       1.1064        -0.0


In [50]:
# Part (ii): Compute predicted probabilities on test data and define metrics function
prob_test = sigmoid(X_test_np @ beta_gd)

def compute_metrics(y_true, prob, threshold):
    y_pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    return {"cm": cm, "TP": tp, "FP": fp, "TN": tn, "FN": fn, 
            "Accuracy": acc, "Precision": prec, "Recall": rec}

In [51]:
# Part (ii): Confusion matrix and metrics for threshold = 0.50
m50 = compute_metrics(y_test, prob_test, 0.50)
print("THRESHOLD = 0.50")
print(f"Confusion Matrix:\n{m50['cm']}")
print(f"TP={m50['TP']}, FP={m50['FP']}, TN={m50['TN']}, FN={m50['FN']}")
print(f"Accuracy: {m50['Accuracy']:.4f}, Precision: {m50['Precision']:.4f}, Recall: {m50['Recall']:.4f}")

THRESHOLD = 0.50
Confusion Matrix:
[[88  0]
 [11  1]]
TP=1, FP=0, TN=88, FN=11
Accuracy: 0.8900, Precision: 1.0000, Recall: 0.0833


In [52]:
# Part (ii): Confusion matrix and metrics for threshold = 0.30
m30 = compute_metrics(y_test, prob_test, 0.30)
print("THRESHOLD = 0.30")
print(f"Confusion Matrix:\n{m30['cm']}")
print(f"TP={m30['TP']}, FP={m30['FP']}, TN={m30['TN']}, FN={m30['FN']}")
print(f"Accuracy: {m30['Accuracy']:.4f}, Precision: {m30['Precision']:.4f}, Recall: {m30['Recall']:.4f}")

THRESHOLD = 0.30
Confusion Matrix:
[[85  3]
 [10  2]]
TP=2, FP=3, TN=85, FN=10
Accuracy: 0.8700, Precision: 0.4000, Recall: 0.1667


In [53]:
# Part (ii): Summary table comparing both thresholds
summary = pd.DataFrame([m50, m30], index=["0.50", "0.30"]).drop(columns="cm")
print(summary)

      TP  FP  TN  FN  Accuracy  Precision    Recall
0.50   1   0  88  11      0.89        1.0  0.083333
0.30   2   3  85  10      0.87        0.4  0.166667


In [54]:
# Part (iv): Estimate Linear Probability Model using OLS
lpm = sm.OLS(y_train, X_train_sm).fit()
print(lpm.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.077
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     4.090
Date:                Fri, 27 Mar 2026   Prob (F-statistic):            0.00139
Time:                        18:53:23   Log-Likelihood:                -56.170
No. Observations:                 250   AIC:                             124.3
Df Residuals:                     244   BIC:                             145.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0856      0.022      3.913      

In [55]:
# Part (iv): Compare Logit and LPM coefficients
comp = pd.DataFrame({"Logit": logit_model.params.values, "LPM": lpm.params.values}, index=names)
print(comp.round(4))

              Logit     LPM
Intercept   -2.5662  0.0856
DTI          0.3624  0.0339
LTV          0.6386  0.0558
Score        0.0617  0.0052
UnempShock   0.7565  0.0726
PriorDelinq  1.1064  0.1521


In [56]:
# Part (iv): Check LPM prediction range (can produce values outside [0,1])
X_test_sm = sm.add_constant(X_test_sc)
prob_lpm = lpm.predict(X_test_sm)
print(f"LPM predictions range: [{prob_lpm.min():.3f}, {prob_lpm.max():.3f}]")
print(f"Predictions < 0: {(prob_lpm < 0).sum()}, Predictions > 1: {(prob_lpm > 1).sum()}")

LPM predictions range: [-0.086, 0.441]
Predictions < 0: 5, Predictions > 1: 0
